# Planet Data API: search, activate, download and preview PlanetScope imagery

**Before you start:** revoke any API key you have posted publicly and create a new one. Planet API keys are credentials used with Basic Authentication on `api.planet.com`. This notebook asks for the key privately with `getpass`.

It uses the **Data API** (not the OAuth-only Processing API). The search filters on geometry, date range and download permission.

## 1. Install packages

In [ ]:
%pip install requests pandas rasterio matplotlib numpy

## 2. Imports

In [ ]:
import json
import time
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
import requests
from requests.auth import HTTPBasicAuth

## 3. Enter your replacement API key
Do not paste the key into a public notebook.

In [ ]:
PLANET_API_KEY = getpass("Enter your new Planet API key: ")

if not PLANET_API_KEY.strip():
    raise RuntimeError("No Planet API key was entered.")

print("API key loaded.")

## 4. Test authentication
The Data API uses HTTP Basic Auth with the API key as username and an empty password.

In [ ]:
AUTH = HTTPBasicAuth(PLANET_API_KEY, "")

response = requests.get("https://api.planet.com/data/v1/", auth=AUTH, timeout=60)
print("HTTP status:", response.status_code)

if not response.ok:
    print(response.text)
    response.raise_for_status()

print("Planet Data API authentication succeeded.")

## 5. Define your AOI

In [ ]:
AOI = {
    "type": "Polygon",
    "coordinates": [[
        [-52.4825, -6.8735],
        [-52.4825, -6.65775],
        [-52.26725, -6.65775],
        [-52.26725, -6.8735],
        [-52.4825, -6.8735]
    ]]
}

print(json.dumps(AOI, indent=2))

## 6. Search parameters
Use dates inside your collection's availability period. If the search returns nothing, run the no-asset-filter cell below.

In [ ]:
ITEM_TYPES = ["PSScene"]

START_DATE = "2022-01-01T00:00:00Z"
END_DATE = "2022-01-31T23:59:59Z"

ASSET_TYPES = ["ortho_analytic_4b_sr"]

SEARCH_URL = "https://api.planet.com/data/v1/quick-search"

print("Search URL:", SEARCH_URL)
print("Date range:", START_DATE, "to", END_DATE)

## 7. Build a downloadable-assets filter
The `PermissionFilter` restricts results to assets your account may download (`assets:download`).

In [ ]:
search_filter = {
    "type": "AndFilter",
    "config": [
        {"type": "GeometryFilter", "field_name": "geometry", "config": AOI},
        {
            "type": "DateRangeFilter",
            "field_name": "acquired",
            "config": {"gte": START_DATE, "lte": END_DATE},
        },
        {"type": "PermissionFilter", "config": ["assets:download"]},
    ],
}

search_payload = {
    "item_types": ITEM_TYPES,
    "asset_types": ASSET_TYPES,
    "filter": search_filter,
}

print(json.dumps(search_payload, indent=2))

## 8. Search Planet

In [ ]:
search_response = requests.post(SEARCH_URL, auth=AUTH, json=search_payload, timeout=120)
print("HTTP status:", search_response.status_code)

if not search_response.ok:
    print(search_response.text)
    search_response.raise_for_status()

search_results = search_response.json()
features = search_results.get("features", [])

print("Search completed.")
print("Returned features:", len(features))

## 9. Search without the asset filter (only if Cell 8 returned zero results)

In [ ]:
if not features:
    payload_no_asset = {"item_types": ["PSScene"], "filter": search_filter}

    r = requests.post(SEARCH_URL, auth=AUTH, json=payload_no_asset, timeout=120)
    print("HTTP status:", r.status_code)

    if not r.ok:
        print(r.text)
        r.raise_for_status()

    search_results = r.json()
    features = search_results.get("features", [])
    print("Returned features:", len(features))
else:
    print("Cell 8 already returned results; skipping.")

## 10. Inspect search results

In [ ]:
if not features:
    print("No downloadable scenes were found.")
else:
    rows = []
    for feature in features:
        p = feature.get("properties", {})
        rows.append({
            "id": feature.get("id"),
            "item_type": p.get("item_type"),
            "acquired": p.get("acquired"),
            "published": p.get("published"),
            "cloud_cover": p.get("cloud_cover"),
            "usable_data": p.get("usable_data"),
        })

    results_df = pd.DataFrame(rows)
    display(results_df.head(20))

## 11. Inspect assets for one scene

In [ ]:
if not features:
    raise RuntimeError(
        "No downloadable scenes were found. Check the collection, dates, "
        "AOI, account permissions, and item type."
    )

first_feature = features[0]
scene_id = first_feature["id"]
item_type = first_feature.get("properties", {}).get("item_type", "PSScene")

print("Scene ID:", scene_id)
print("Item type:", item_type)

assets_url = f"https://api.planet.com/data/v1/item-types/{item_type}/items/{scene_id}/assets"

assets_response = requests.get(assets_url, auth=AUTH, timeout=60)
print("HTTP status:", assets_response.status_code)

if not assets_response.ok:
    print(assets_response.text)
    assets_response.raise_for_status()

assets = assets_response.json()

print("Available assets:")
for name, info in assets.items():
    print(name, "| status:", info.get("status"), "| type:", info.get("type"))

## 12. Select a downloadable asset
Prefers surface-reflectance analytic imagery when available.

In [ ]:
def asset_is_downloadable(asset_info):
    # True when the asset has a download URL or can be activated.
    return bool(
        asset_info.get("location")
        or asset_info.get("href")
        or asset_info.get("_links", {}).get("activate")
    )


downloadable_assets = {n: i for n, i in assets.items() if asset_is_downloadable(i)}

print("Downloadable or activatable assets:")
for name in downloadable_assets:
    print("-", name)

if not downloadable_assets:
    raise RuntimeError("No downloadable assets were returned for this scene.")

preferred_asset_names = [
    "ortho_analytic_4b_sr",
    "ortho_analytic_8b_sr",
    "ortho_analytic_4b",
    "ortho_analytic_8b",
]

selected_asset_name = next(
    (n for n in preferred_asset_names if n in downloadable_assets),
    next(iter(downloadable_assets)),
)

selected_asset_info = assets[selected_asset_name]

print("\nSelected asset:", selected_asset_name)
print(json.dumps(selected_asset_info, indent=2)[:3000])

## 13. Activate the asset if necessary

In [ ]:
activate_url = selected_asset_info.get("_links", {}).get("activate")
asset_status = selected_asset_info.get("status")

print("Asset status:", asset_status)
print("Activation URL exists:", bool(activate_url))

if activate_url and asset_status != "active":
    activate_response = requests.get(activate_url, auth=AUTH, timeout=120)
    print("Activation HTTP status:", activate_response.status_code)

    if not activate_response.ok:
        print(activate_response.text)
        activate_response.raise_for_status()

    print("Activation request sent.")
else:
    print("Asset is already active or does not require activation.")

## 14. Wait until the asset is active and get the download URL
Polls every 10 seconds for up to 15 minutes.

In [ ]:
download_url = None

for attempt in range(90):
    r = requests.get(assets_url, auth=AUTH, timeout=60)
    r.raise_for_status()
    selected_asset_info = r.json()[selected_asset_name]

    status = selected_asset_info.get("status")
    download_url = selected_asset_info.get("location") or selected_asset_info.get("href")
    print(f"Attempt {attempt + 1}: status = {status}")

    if status == "active" and download_url:
        break
    time.sleep(10)

if not download_url:
    raise RuntimeError(
        "The asset is not ready yet. Wait a bit and rerun this cell."
    )

print("Download URL found.")

## 15. Download the asset

In [ ]:
OUTPUT_DIR = Path("planet_downloads")
OUTPUT_DIR.mkdir(exist_ok=True)

safe_asset_name = selected_asset_name.replace("/", "_")
output_file = OUTPUT_DIR / f"{scene_id}_{safe_asset_name}.tif"

# The location URL is pre-signed, so no Planet credentials are sent with it.
with requests.get(download_url, stream=True, timeout=600) as download_response:
    print("HTTP status:", download_response.status_code)

    if not download_response.ok:
        print(download_response.text[:2000])
        download_response.raise_for_status()

    with open(output_file, "wb") as f:
        for chunk in download_response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Downloaded file:", output_file)
print("File size:", f"{output_file.stat().st_size:,}", "bytes")

## 16. Read the GeoTIFF

In [ ]:
import rasterio

with rasterio.open(output_file) as src:
    print("Driver:", src.driver)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bands:", src.count)
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)
    print("Data type:", src.dtypes)
    image = src.read()

print("Array shape:", image.shape)

## 17. Display the image

In [ ]:
import matplotlib.pyplot as plt


def scale_band(band):
    band = band.astype("float32")
    valid = np.isfinite(band)

    if not valid.any():
        return np.zeros_like(band, dtype="float32")

    low, high = np.percentile(band[valid], [2, 98])

    if high <= low:
        return np.zeros_like(band, dtype="float32")

    return np.clip((band - low) / (high - low), 0, 1)


plt.figure(figsize=(10, 8))

if image.shape[0] >= 4:
    # 4-band PlanetScope order: blue, green, red, nir
    blue, green, red = scale_band(image[0]), scale_band(image[1]), scale_band(image[2])
    plt.imshow(np.dstack([red, green, blue]))
    plt.title(f"{scene_id} - RGB preview")
else:
    plt.imshow(image[0], cmap="gray")
    plt.title(f"{scene_id} - single-band preview")

plt.axis("off")
plt.show()

## 18. Calculate NDVI
Requires red and near-infrared bands. Surface-reflectance assets are scaled integers (divide by 10,000), but NDVI is a ratio so scaling cancels out.

In [ ]:
if image.shape[0] >= 4:
    red = image[2].astype("float32")
    nir = image[3].astype("float32")
    denominator = nir + red

    ndvi = np.where(denominator != 0, (nir - red) / denominator, np.nan)

    print("NDVI minimum:", np.nanmin(ndvi))
    print("NDVI maximum:", np.nanmax(ndvi))

    plt.figure(figsize=(10, 8))
    plt.imshow(ndvi, cmap="RdYlGn", vmin=-1, vmax=1)
    plt.colorbar(label="NDVI")
    plt.title(f"{scene_id} - NDVI")
    plt.axis("off")
    plt.show()
else:
    print("This asset has fewer than 4 bands; NDVI skipped.")

## If the search returns zero scenes

1. Confirm the old API key was revoked and the new one is in use.
2. Check the date range is within the collection's actual dates.
3. Confirm your account has download access to the collection.
4. Remove the asset filter (Cell 9).
5. Check the item type matches the collection.
6. Inspect the raw response:

```python
print(json.dumps(search_response.json(), indent=2)[:5000])
```